[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 中等：二维卷积

从零实现 **二维卷积**。

### 函数签名
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # 返回: (B, C_out, H_out, W_out)
```

### 规则
- 不要使用 `F.conv2d` 或 `nn.Conv2d`
- 支持 `stride` 和 `padding` 参数
- 允许使用 `F.pad` 进行零填充

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✏️ 在此实现你的代码

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    pass  # 提取补丁，应用卷积核，处理 stride/padding

命名:

H = Height（高度）\
W = Width（宽度）\
C = Channels（通道数）\
B = Batch size（批次大小）\
kH = Kernel Height（卷积核高度）\
kW = Kernel Width（卷积核宽度）

In [ ]:
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    """
    从零实现二维卷积，先最输入进行填充，然后创建一个输出形状的张量，遍历 kernal 形状的参数和输入相乘并求和得到输出的每一个参数
    
    Args:
        x: (B, C_in, H, W) 输入张量
        weight: (C_out, C_in, kH, kW) 卷积核权重
        bias: (C_out,) 偏置项，可选
        stride: 步长
        padding: 零填充大小
    
    Returns:
        (B, C_out, H_out, W_out) 输出张量
    """
    # 获取输入和卷积核的维度
    B, C_in, H, W = x.shape
    C_out, C_in_k, kH, kW = weight.shape
    assert C_in == C_in_k, "输入通道数与卷积核通道数不匹配"
    
    # pad 参数表示从最后一个维度开始，依次指定每个维度左边和右边的填充数量。
    if padding > 0:
        x_padded = F.pad(x, pad=(padding, padding, padding, padding), mode='constant', value=0)
    else:
        x_padded = x
    
    # **计算输出尺寸**
    H_out = (H + 2 * padding - kH) // stride + 1
    W_out = (W + 2 * padding - kW) // stride + 1
    
    # 初始化输出张量
    output = torch.zeros(B, C_out, H_out, W_out)
    
    # 对批次中的每个样本(input_batch)
    for b in range(B):
        # 对每个输出通道(out_channel)
        for c_out in range(C_out):
            # 对每个输出位置(out_HW)
            for i in range(H_out):
                for j in range(W_out):
                    # 计算输入窗口的起始位置, 每步 stride 长度
                    h_start = i * stride
                    w_start = j * stride
                    
                    # 提取输入窗口
                    window = x_padded[b, :, h_start:h_start+kH, w_start:w_start+kW]
                    
                    # 卷积核与窗口逐元素相乘并求和
                    # weight[c_out] shape: (C_in, kH, kW)
                    # window shape: (C_in, kH, kW)
                    output[b, c_out, i, j] = torch.sum(weight[c_out] * window)
            
            # 添加偏置
            if bias is not None:
                output[b, c_out] += bias[c_out]
    
    return output

In [ ]:
# 🧪 调试
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('输出:', my_conv2d(x, w).shape)
print('匹配:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

In [ ]:
# ✅ 提交
from torch_judge import check
check('conv2d')